In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

In [ ]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess



In [ ]:
import sys

sys.path.append('..')

from kranocyte.io import load_full_adata, add_ensembl_ids
from kranocyte.dropletQC import classify_empty_and_damaged
from kranocyte.general import preprocessing_adata_sub
from kranocyte.plot_functions import magma, set_plotting_style, plot_volcano, plot_cell_stats, plot_gene_stats, savefig
from kranocyte.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [ ]:
from kranocyte.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from kranocyte.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from kranocyte.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from kranocyte.characterization import subset_and_reprocess, check_marker_dict, population_composition, reprocess_in_place
from kranocyte.cell_types import DICT_RENAMING_INTEGRATION, PALETTE_CELL_TYPE_INTEGRATION

In [ ]:

from kranocyte.interactions import run_liana, filter_liana, annotate_liana, plot_liana

In [ ]:
DATA_DIR_AA = f"{BASE_DIR}/data/ARAUZO_03/"
DATA_DIR_NI = f"{BASE_DIR}/data/public_scRNAseq/datasets/nicoletti_2023_GSE221736"
DATA_DIR_SO = f"{BASE_DIR}/data/public_scRNAseq/datasets/song_2023_GSE215922"
DATA_DIR_SU = f"{BASE_DIR}/data/public_scRNAseq/datasets/southerland_2023_GSE227075"


In [ ]:
GRUPO = "cell_type"     # the partition on which PAGA is defined
LOTE  = "gsm"

# Globins: 45% of the transcriptome in some samples, and 100% spliced.
# Excluded from velocity_genes, no exceptions.
HB = ["Hbb-bs", "Hba-a1", "Hba-a2", "Hbb-bt", "Alas2", "Ahsp", "Bpgm"]

In [ ]:
from datetime import date
TODAY = str(date.today())

FIG_DIR = f"{BASE_DIR}/figures/{TODAY}/"
RESULTS_DIR = f"{BASE_DIR}/results/{TODAY}/"

In [ ]:
adata_AA = sc.read(f"{DATA_DIR_AA}/processed_adatas/AA_inhouse_dataset_processed.h5ad")
adata_NI = sc.read(f"{DATA_DIR_NI}/processed_adatas/NI_nicoletti_processed.h5ad")
adata_SO = sc.read(f"{DATA_DIR_SO}/processed_adatas/SO_song_processed.h5ad")
adata_SU = sc.read(f"{DATA_DIR_SU}/processed_adatas/SU_southerland_processed.h5ad")


In [ ]:
# We filter to get only the control data, we are not interested in the rest.

for adata in adata_AA, adata_NI, adata_SO, adata_SU:
    print(adata.obs["condition"].cat.categories)

adata_NI = adata_NI[adata_NI.obs["condition"] == "non_DEN"]
adata_SO = adata_SO[adata_SO.obs["condition"] == "sham"]
adata_SU = adata_SU[adata_SU.obs["condition"] == "C57BL6_sham"]

In [ ]:


# We apply this 
DICT_RENAMING_INTEGRATION = {
    'FAP_A': 'FAP.1',
    'FAP_AB': 'FAP.1',
    'FAP_AF': 'FAP.1',
    'FAP_B1': 'FAP.4',    
    'FAP_B2': 'FAP.4',
    'FAP_E': 'FAP.2',
    'FAP_C': 'FAP.3',
    'FAP_D': 'FAP.3',
    'FAP_F': 'FAP.3',
    'Krano_A': 'FAP.5',
    'Krano_B': 'FAP.6',
    'Krano_C': 'FAP.6',
    'Endothelial*': 'ENDO',
    'Sat_A1': 'SAT', 
    'Sat_A2': 'SAT', 
    'Sat_B': 'SAT', 
    'Teno_A': 'TNMD', 
    'Teno_D': 'TNMD',
    'Teno_B': 'TNMD', 
    'Teno_C': 'TNMD', 

    # Other populations
    "Immune": "IMM.MONO-MAC", 
    "B_cell": "IMM.B", 
    "Dendritic": "IMM.DEN",
    "Mast_cell": "IMM.MAST",
    "T_NK": "IMM.NK",
    "Neutrophil": "IMM.NEU",

    'Sat_U': 'SAT', 
    'Krano_U': 'FAP.5',
    "SMC": "SMC-SMMC",
    "SMMC": "SMC-SMMC",
    "Glial_Schwann": "GLIA",
    "Lymphatic_EC": "ENDO.LYMPH",
    'Pericyte': 'ENDO.PERI',
                               
}




PALETTE_CELL_TYPE_INTEGRATION = {
    # ---- Minor (más oscuros) ----
    # FAP (7)
    'FAP.1':"#A1D9F0",
    'FAP.2' :"#4C92AD",
    'FAP.3':"#12485E",
    'FAP.4':"#B2A1F0",
    'FAP.5' :"#694CAD",
    'FAP.6':"#26125E",

    "TNMD":"#6F5214",
    'SAT':"#2E9277",
    'SMC-SMMC':      "#79DD6F",
    'GLIA':      "#CAC031",

    'ENDO':   "#C23C64",
    'ENDO.LYMPH':   "#F8608D",
    'ENDO.PERI':   "#991F43",

    'IMM.MONO-MAC':      "#331903",
    'IMM.B':      "#D36B17",
    'IMM.NK':      "#F7A765",
    'IMM.DEN':      "#96490A",
    'IMM.MAST':      "#D35217",
    'IMM.NEU':      "#92390F",
}

In [ ]:
for adata in adata_AA, adata_NI, adata_SO, adata_SU:
    adata.obs["cell_type"] = adata.obs["minor_population"].astype(str).replace(DICT_RENAMING_INTEGRATION)
    adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
    adata.uns["cell_type_colors"] = [PALETTE_CELL_TYPE_INTEGRATION[i] for i in adata.obs["cell_type"].cat.categories]


In [ ]:
sc.pl.umap(adata_AA, color="cell_type")
sc.pl.umap(adata_NI, color="cell_type")
sc.pl.umap(adata_SO, color="cell_type")
sc.pl.umap(adata_SU, color="cell_type")

In [ ]:
adata_joint = sc.AnnData.concatenate(*[adata_AA, adata_NI, adata_SO, adata_SU], 
                                     batch_key="dataset", 
                                     batch_categories=["Fuertes", "Nicoletti", "Song", "Southerland"] )

In [ ]:
reprocess_in_place(adata_joint, batch_key="gsm", n_comps=40, )

In [ ]:
adata_joint.uns["cell_type_colors"] = [PALETTE_CELL_TYPE_INTEGRATION[i] for i in adata_joint.obs["cell_type"].cat.categories]
sc.pl.umap(adata_joint, color=["dataset",])
sc.pl.umap(adata_joint, color=["cell_type"])

In [ ]:
sc.pl.umap(adata_joint, color=["cell_type"], legend_loc="on data", legend_fontsize=4, legend_fontoutline=2)

In [ ]:
adata_joint.obs["cell_type"].value_counts()

# Interaction table

We are going to use the whole table of interactions and then filter by type of interaction.

In [ ]:
liana_df  = annotate_liana(run_liana(adata_joint, "cell_type", n_perms=3000, n_jobs=20), adata_joint, "cell_type")

In [ ]:
liana_df_AA  = annotate_liana(run_liana(adata_AA, "cell_type", n_perms=3000, n_jobs=20), adata_AA, "cell_type")

In [ ]:
liana_df_NI  = annotate_liana(run_liana(adata_NI, "cell_type", n_perms=3000, n_jobs=20), adata_NI, "cell_type")

In [ ]:
liana_df_SO  = annotate_liana(run_liana(adata_SO, "cell_type", n_perms=3000, n_jobs=20), adata_SO, "cell_type")

In [ ]:
liana_df_SU  = annotate_liana(run_liana(adata_SU, "cell_type", n_perms=3000, n_jobs=20), adata_SU, "cell_type")

## FAP - others

In [ ]:
FAP_POPS = ["FAP.1", "FAP.2", "FAP.3", "FAP.4", "FAP.5", "FAP.6"]

def filter_liana_pops(df, filter_type="FAP_FAP"): # FAP_FAP or FAP_OTHER
    if filter_type == "FAP_OTHER":
        return df[(df["source"].isin(FAP_POPS) & ~ df["target"].isin(FAP_POPS))  | (~ df["source"].isin(FAP_POPS) & df["target"].isin(FAP_POPS)) ].sort_values(by="spec_weight_oe", ascending=False)
    elif filter_type == "FAP_FAP":
        return df[(df["source"].isin(FAP_POPS)) & (df["target"].isin(FAP_POPS))].sort_values(by="spec_weight_oe", ascending=False)

In [ ]:
liana_df_FAP_others = filter_liana_pops(liana_df, filter_type="FAP_OTHER")
liana_df_FAP_others_AA = filter_liana_pops(liana_df_AA, filter_type="FAP_OTHER")
liana_df_FAP_others_NI = filter_liana_pops(liana_df_NI, filter_type="FAP_OTHER")
liana_df_FAP_others_SO = filter_liana_pops(liana_df_SO, filter_type="FAP_OTHER")
liana_df_FAP_others_SU = filter_liana_pops(liana_df_SU, filter_type="FAP_OTHER")

In [ ]:
"""Dotplots for FAP populations and for arbitrary cell groups."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FAP_POPS = [f"FAP.{i}" for i in range(1, 7)]


def make_lr_label(row, focal_populations=None):
    focal = FAP_POPS if focal_populations is None else focal_populations
    ligand = str(row["ligand_complex"])
    receptor = str(row["receptor_complex"])
    if row["source"] in focal:
        ligand = r"$\mathbf{" + ligand.replace("_", r"\_") + "}$"
    if row["target"] in focal:
        receptor = r"$\mathbf{" + receptor.replace("_", r"\_") + "}$"
    return f"{ligand} → {receptor}"


def get_fap_direction(row):
    source_is_fap = row["source"] in FAP_POPS
    target_is_fap = row["target"] in FAP_POPS
    if source_is_fap and target_is_fap:
        return "FAP_to_FAP"
    if source_is_fap:
        return "FAP_to_other"
    if target_is_fap:
        return "other_to_FAP"
    return "other"


def _population_names(value):
    names = [value] if isinstance(value, str) else list(value)
    if not names or any(not isinstance(name, str) or not name for name in names):
        raise ValueError("Give one or more non-empty population names.")
    return list(dict.fromkeys(names))


def _prepare(df_liana, population, specificity_rank_thresh,
             magnitude_rank_thres, spec_weight_oe_thresh, lr_logfc_thresh,
             focal_populations=None):
    required = ["source", "target", "ligand_complex", "receptor_complex",
                "spec_weight_oe", "specificity_rank", "magnitude_rank", "lr_logfc"]
    missing = [column for column in required if column not in df_liana]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    generic = focal_populations is not None
    focal = _population_names(focal_populations) if generic else FAP_POPS
    if generic:
        if population is None:
            raise ValueError("With focal_populations, name the partner groups in population.")
        partners = _population_names(population)
        if set(focal) & set(partners):
            raise ValueError("Focal and partner groups must be disjoint; for FAP->FAP use population='FAP'.")
        mask = ((df_liana["source"].isin(focal) & df_liana["target"].isin(partners))
                | (df_liana["source"].isin(partners) & df_liana["target"].isin(focal)))
        between_faps = False
    elif population is not None: # to avoid double filtering
        source_fap = df_liana["source"].isin(FAP_POPS)
        target_fap = df_liana["target"].isin(FAP_POPS)
        between_faps = population == "FAP" or population in FAP_POPS
        if between_faps:
            mask = source_fap & target_fap
            if population != "FAP":
                mask &= (df_liana["source"].eq(population)
                        | df_liana["target"].eq(population))
        else:
            mask = ((source_fap & df_liana["target"].eq(population))
                    | (target_fap & df_liana["source"].eq(population)))
    else:
        between_faps = False
        mask = pd.Series(True, index=df_liana.index)
        
    # Copying before assigning columns avoids modifying views of the original.
    frame = df_liana.loc[mask].copy()
    if frame[["source", "target", "ligand_complex", "receptor_complex"]].isna().any().any():
        raise ValueError("Some interaction identities are incomplete.")
    for column in ["source", "target", "ligand_complex", "receptor_complex"]:
        frame[column] = frame[column].astype(str)

    frame["log_spec_weight_oe"] = np.log10(
        frame["spec_weight_oe"].where(frame["spec_weight_oe"] > 0)
    )
    frame["specificity_score"] = 1 - frame["specificity_rank"]
    frame["magnitude_score"] = 1 - frame["magnitude_rank"]

    if generic:
        source_is_focal = frame["source"].isin(focal)
        frame["direction"] = np.where(source_is_focal, "focal_to_partner", "partner_to_focal")
        # Keeping the historical column name lets the same plot be reused.
        frame["FAP_population"] = frame["source"] + " → " + frame["target"]
        x_order = [pair for partner in partners for cell in focal
                   for pair in (f"{cell} → {partner}", f"{partner} → {cell}")]
    elif between_faps:
        frame["direction"] = "FAP_to_FAP"
        # Each direction is its own column, the autocrine one included.
        frame["FAP_population"] = frame["source"] + " → " + frame["target"]
        present_pairs = set(frame["FAP_population"])
        x_order = [f"{source} → {target}"
                   for source in FAP_POPS for target in FAP_POPS
                   if f"{source} → {target}" in present_pairs]
    else:
        source_is_fap = frame["source"].isin(FAP_POPS)
        frame["direction"] = np.where(source_is_fap, "FAP_to_other", "other_to_FAP")
        frame["FAP_population"] = frame["source"].where(source_is_fap, frame["target"])
        x_order = FAP_POPS

    frame["LR_name"] = frame["ligand_complex"] + "-" + frame["receptor_complex"]
    # One identifier per LR across every column being compared.
    frame["LR_id"] = frame["LR_name"] + "__" + frame["direction"]
    frame["LR_label"] = pd.Series(
        [make_lr_label(row, focal) for _, row in frame.iterrows()],
        index=frame.index, dtype=object,
    )
    frame["FAP_population"] = pd.Categorical(
        frame["FAP_population"], categories=x_order, ordered=True
    )
    frame["is_interesting"] = (
        (frame["specificity_rank"] < specificity_rank_thresh)
        & (frame["magnitude_rank"] < magnitude_rank_thres)
        & (frame["spec_weight_oe"] > spec_weight_oe_thresh)
        & (frame["lr_logfc"] > lr_logfc_thresh)
    ).fillna(False)
    return frame


def get_df_pop(df_liana, population, specificity_rank_thresh=0.05,
               magnitude_rank_thres=0.05, spec_weight_oe_thresh=10,
               lr_logfc_thresh=0.5, circle_size_var="spec_weight_oe", *,
               focal_populations=None):
    """Select the LR pairs that pass the filters in at least one column and keep
    all of their available columns so they can be compared. Returns the same
    three objects as the original function: df_plot, label_map, lr_order.
    focal_populations=None keeps the FAP mode. For other groups, for example:
    population=['ENDO', 'ENDO.PERI'], focal_populations=['GLIA.tSC'].
    The generic mode puts directed pairs on X; genes of the focal population
    are shown in bold. Sending and receiving get different LR_id values.
    """
    frame = _prepare(df_liana, population, specificity_rank_thresh,
                     magnitude_rank_thres, spec_weight_oe_thresh, lr_logfc_thresh,
                     focal_populations=focal_populations)
    if circle_size_var not in frame:
        raise ValueError(f"No such metric: {circle_size_var!r}.")
    interesting_ids = frame.loc[frame["is_interesting"], "LR_id"].unique()
    # Select by ID, not by the visible label: sending and receiving differ.
    frame = frame.loc[frame["LR_id"].isin(interesting_ids)].copy()
    lr_order = (
        frame.groupby("LR_id", observed=True)[circle_size_var]
        .agg(["max", "mean"])
        .sort_values(["max", "mean"], ascending=False, kind="stable")
        .index.tolist()
    )
    frame["LR_id"] = pd.Categorical(frame["LR_id"], categories=lr_order, ordered=True)
    label_map = (frame[["LR_id", "LR_label"]].drop_duplicates()
                 .set_index("LR_id")["LR_label"])
    return frame, label_map, lr_order


def get_df_pop_fixed_lr(df_liana, population, lr_order,
                        specificity_rank_thresh=0.05, magnitude_rank_thres=0.05,
                        spec_weight_oe_thresh=10, lr_logfc_thresh=0.5, *,
                        focal_populations=None):
    """Keep the LR pairs and their order from an earlier selection in the same mode.
    The filters only recompute the asterisks; they do not select LR pairs again.
    """
    frame = _prepare(df_liana, population, specificity_rank_thresh,
                     magnitude_rank_thres, spec_weight_oe_thresh, lr_logfc_thresh,
                     focal_populations=focal_populations)
    lr_order = list(lr_order)
    frame = frame.loc[frame["LR_id"].isin(lr_order)].copy()
    frame["LR_id"] = pd.Categorical(frame["LR_id"], categories=lr_order, ordered=True)
    return frame




def plot_liana_pop(df_plot, label_map, lr_order,
                   circle_size_var="spec_weight_oe", circle_hue_var="lr_logfc",
                   size_lim=(5, 500), hue_lim=(-4, 4), figsize=None,
                   shade_fap_blocks=True, block_color="0.94"):
    """Draw the prepared table. Numeric coordinates are fixed so that the
    asterisks and the ticks line up even when intermediate categories are missing.
    In FAP -> FAP it shades the blocks of FAP.1, FAP.3 and FAP.5 as senders.
    The shading follows the identities even if a dataset lacks some pairs.
    """
    lr_order = list(lr_order)
    if len(set(lr_order)) != len(lr_order):
        raise ValueError("lr_order contains repeated IDs.")
    if isinstance(df_plot["FAP_population"].dtype, pd.CategoricalDtype):
        x_order = df_plot["FAP_population"].cat.categories.tolist()
    else:
        x_order = df_plot["FAP_population"].dropna().unique().tolist()
    frame = df_plot.copy()
    if frame.duplicated(["FAP_population", "LR_id"]).any():
        raise ValueError(
            "There are several rows for the same LR and column. Split the datasets "
            "or samples before drawing; the points would overlap."
        )
    frame["_plot_x"] = frame["FAP_population"].astype(object).map(
        {name: i for i, name in enumerate(x_order)}
    )
    frame["_plot_y"] = frame["LR_id"].astype(object).map(
        {name: i for i, name in enumerate(lr_order)}
    )
    if frame[["_plot_x", "_plot_y"]].isna().any().any():
        raise ValueError("Some categories are absent from the axis ordering.")
    if figsize is None:
        figsize = (max(5, 0.48 * len(x_order) + 2), 12)
    fig, ax = plt.subplots(figsize=figsize)
    if shade_fap_blocks:
        shaded_sources = set(FAP_POPS[::2])
        # Grouping consecutive columns keeps the bands even if the user
        # reorders categories or some directed pairs are missing.
        start = 0
        while start < len(x_order):
            source, separator, target = str(x_order[start]).partition(" → ")
            if separator and source in shaded_sources and target in FAP_POPS:
                end = start + 1
                while end < len(x_order):
                    next_source, next_sep, next_target = str(x_order[end]).partition(" → ")
                    if not (next_sep and next_source == source and next_target in FAP_POPS):
                        break
                    end += 1
                ax.axvspan(start - 0.5, end - 0.5, facecolor=block_color,
                           edgecolor="none", zorder=0)
                start = end
            else:
                start += 1
    if not frame.empty:
        # Seaborn drops points with no size/colour; do not draw their asterisk either.
        valid = np.isfinite(frame[circle_size_var]) & np.isfinite(frame[circle_hue_var])
        visible = frame.loc[valid]
        if not visible.empty:
            sns.scatterplot(
                data=visible, x="_plot_x", y="_plot_y",
                size=circle_size_var, hue=circle_hue_var, sizes=size_lim,
                palette="RdBu_r", hue_norm=hue_lim,
                edgecolor="black", linewidth=0.3, ax=ax,
            )
            for _, row in visible.loc[visible["is_interesting"]].iterrows():
                ax.annotate("*", (row["_plot_x"], row["_plot_y"]),
                            xytext=(0, -2), textcoords="offset points",
                            ha="center", va="center", fontsize=8,
                            fontweight="bold", color="black", zorder=10)
            ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
                      borderaxespad=0, frameon=False)
    else:
        ax.text(0.5, 0.5, "No interactions to show",
                transform=ax.transAxes, ha="center", va="center")

    ax.set_xticks(range(len(x_order)), labels=x_order, rotation=90,)
    ax.set_yticks(range(len(lr_order)),
                  labels=[label_map.get(lr_id, lr_id) for lr_id in lr_order])
    ax.set_xlim(-0.5, max(len(x_order) - 0.5, 0.5))
    ax.set_ylim(max(len(lr_order) - 0.5, 0.5), -0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.tight_layout()
    return fig

def _plot_title(population, focal_populations):
    if focal_populations is None:
        return f"L-R interactions between FAPs and {population}"
    return ("L-R interactions: " + ", ".join(_population_names(focal_populations))
            + " ↔ " + ", ".join(_population_names(population)))


def plot_general_case(df_liana, population, specificity, magnitude, spec_weight, lr_logfc, 
                      circle_size_var, circle_hue_var, 
                      size_lim=(5, 400), hue_lim=(-4, 4), figsize=(4, 12), ytitle=0.93, *,
                      focal_populations=None):
    
    df_plot, label_map, lr_order = get_df_pop(df_liana, population, 
                                                specificity_rank_thresh=specificity,
                                                magnitude_rank_thres=magnitude, 
                                                spec_weight_oe_thresh=spec_weight, 
                                                lr_logfc_thresh=lr_logfc, 
                                                circle_size_var=circle_size_var,
                                                focal_populations=focal_populations)
    fig = plot_liana_pop(df_plot, label_map, lr_order, 
                        circle_size_var=circle_size_var, circle_hue_var=circle_hue_var, 
                        size_lim=size_lim, hue_lim=hue_lim, figsize=figsize)
    fig.suptitle(_plot_title(population, focal_populations), y=ytitle)

    return df_plot, label_map, lr_order, fig


def plot_specific_case(df_liana_dataset, dataset_name, population, lr_order, label_map,
                       specificity, magnitude, spec_weight, lr_logfc, 
                       circle_size_var, circle_hue_var, 
                       size_lim_general=(5, 400), hue_lim_general=(-4, 4), figsize_general=(4, 12), ytitle_general=0.93, 
                       size_lim_specific=(5, 400), hue_lim_specific=(-4, 4), figsize_specific=(4, 12), ytitle_specific=0.93, *,
                       focal_populations=None):
    
    df_plot_dataset_general = get_df_pop_fixed_lr(
        df_liana_dataset,
        population=population,
        lr_order=lr_order,
        specificity_rank_thresh=specificity,
        magnitude_rank_thres=magnitude,
        spec_weight_oe_thresh=spec_weight,
        lr_logfc_thresh=lr_logfc,
        focal_populations=focal_populations,
        )
    fig_general = plot_liana_pop(df_plot_dataset_general, label_map, lr_order, 
                        circle_size_var=circle_size_var, circle_hue_var=circle_hue_var, 
                        size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general)
    title = _plot_title(population, focal_populations)
    fig_general.suptitle(f"{title} - {dataset_name}\nTop global interactions", y=ytitle_general)


    
    df_plot_dataset_specific, label_map_dataset, lr_order_dataset = get_df_pop(df_liana_dataset, 
                                                                               population=population, 
                                                specificity_rank_thresh=specificity,
                                                magnitude_rank_thres=magnitude, 
                                                spec_weight_oe_thresh=spec_weight, 
                                                lr_logfc_thresh=lr_logfc, circle_size_var=circle_size_var,
                                                focal_populations=focal_populations)
    fig_specific = plot_liana_pop(df_plot_dataset_specific, label_map_dataset, lr_order_dataset, 
                        circle_size_var=circle_size_var, circle_hue_var=circle_hue_var, 
                        size_lim=size_lim_specific, hue_lim=hue_lim_specific, figsize=figsize_specific)
    fig_specific.suptitle(f"{title} - {dataset_name}\nTop specific interactions", y=ytitle_specific)

    return df_plot_dataset_specific, label_map_dataset, lr_order_dataset, fig_general, fig_specific


In [ ]:
liana_df_FAP_others

In [ ]:
# plot_general_case y plot_specific_case están definidas junto a plot_liana_pop en la celda anterior.

In [ ]:
dict_datasets = {"Fuertes": liana_df_FAP_others_AA, 
                 "Nicoletti": liana_df_FAP_others_NI, 
                 "Song": liana_df_FAP_others_SO, 
                 "Southerland": liana_df_FAP_others_SU}

CIRCLE_SIZE = "spec_weight_oe"
CIRCLE_COLOR = "lr_logfc"




In [ ]:
adata_joint.obs["cell_type"].cat.categories

In [ ]:
filter_liana(liana_df_FAP_others, genes=["Mag"]).sort_values("spec_weight_oe")

In [ ]:
POP_CHOICE = "ENDO"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=30
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    print(dataset)
    df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                        SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                        CIRCLE_SIZE, CIRCLE_COLOR, 
                        size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                        size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                        figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                        ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

    savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    plt.show()



In [ ]:
POP_CHOICE = "ENDO.LYMPH"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.9
MAGNITUDE=0.9
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "ENDO.PERI"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.7
MAGNITUDE=0.7
SPEC_WEIGHT=20
LR_LOGFC=0.7


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]


fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "GLIA"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.B"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 6)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.DEN"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1



# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.MAST"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1

# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.MONO-MAC"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.NEU"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.NK"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1

# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "SAT"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(6, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (6, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (6, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (6, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (6, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "SMC-SMMC"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "TNMD"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1

# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
df_plot

## FAP - others

In [ ]:
liana_df_FAP_FAP = filter_liana_pops(liana_df, filter_type="FAP_FAP")
liana_df_FAP_FAP_AA = filter_liana_pops(liana_df_AA, filter_type="FAP_FAP")
liana_df_FAP_FAP_NI = filter_liana_pops(liana_df_NI, filter_type="FAP_FAP")
liana_df_FAP_FAP_SO = filter_liana_pops(liana_df_SO, filter_type="FAP_FAP")
liana_df_FAP_FAP_SU = filter_liana_pops(liana_df_SU, filter_type="FAP_FAP")

In [ ]:
POP_CHOICE = "FAP"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(18, 16)
ytitle_general=1

dict_figparams = {"Fuertes": {"figsize_specific": (18, 16), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (18, 16), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (18, 16), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (18, 16), "ytitle_specific": 0.99}, }

SPECIFICITY=0.85
MAGNITUDE=0.85
SPEC_WEIGHT=15
LR_LOGFC=0.7



# Tune in filtering parameters
df_liana_pop = liana_df_FAP_FAP
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_FAP, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

display(df_plot)

# Apply to the populations
for dataset, liana_df_FAP_FAP_dataset in dict_datasets.items():
    print(dataset)
    df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_FAP_dataset, dataset, POP_CHOICE, lr_order, label_map,
                        SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                        CIRCLE_SIZE, CIRCLE_COLOR, 
                        size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                        size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                        figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                        ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

    savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    plt.show()



In [ ]:
liana_df_FAP_FAP

## LIANA of specific populations

In [ ]:
POP_CHOICE = "ENDO"

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=20
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]

df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


list_dfs, list_label_map, list_lr_order = [], [], []

SPEC_GENES_1 = ["Flt1", "Kdr", "Tie1"]

for gene in SPEC_GENES_1:
    df_liana_pop_gene = df_liana_pop[(df_liana_pop["ligand_complex"].str.contains(gene)) | (df_liana_pop["receptor_complex"].str.contains(gene))]
    df_plot, label_map, lr_order = get_df_pop(df_liana_pop_gene, POP_CHOICE, 
                                                specificity_rank_thresh=SPECIFICITY,
                                                magnitude_rank_thres=MAGNITUDE, 
                                                spec_weight_oe_thresh=SPEC_WEIGHT, 
                                                lr_logfc_thresh=LR_LOGFC, 
                                                circle_size_var=CIRCLE_SIZE,)

    list_dfs.append(df_plot)
    list_label_map.append(label_map)
    list_lr_order += [i for i in lr_order if i not in list_lr_order]    

SPEC_GENES_2 = ["Selp", "Aqp"]


for gene in SPEC_GENES_2:
    df_liana_pop_gene = df_liana_pop[(df_liana_pop["ligand_complex"].str.contains(gene)) | (df_liana_pop["receptor_complex"].str.contains(gene))]
    df_plot, label_map, lr_order = get_df_pop(df_liana_pop_gene, POP_CHOICE, 
                                                specificity_rank_thresh=SPECIFICITY,
                                                magnitude_rank_thres=MAGNITUDE, 
                                                spec_weight_oe_thresh=SPEC_WEIGHT, 
                                                lr_logfc_thresh=LR_LOGFC, 
                                                circle_size_var=CIRCLE_SIZE,)

    list_dfs.append(df_plot)
    list_label_map.append(label_map)
    list_lr_order += [i for i in lr_order if i not in list_lr_order]    
    

plot_liana_pop(df_plot = pd.concat(list_dfs).drop_duplicates(),
                label_map = pd.concat(list_label_map).drop_duplicates(),
                lr_order = list_lr_order, 
                figsize=(6, 6)            )

plt.gca().axhline(y=10.5, color='#bcbcbc', linewidth=1)
fig = plt.gcf()

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}_SPECIFIC_GENES", fig_dir=FIG_DIR)


In [ ]:
POP_CHOICE = "ENDO.PERI"

SPECIFICITY=0.8
MAGNITUDE=0.9
SPEC_WEIGHT=20
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]

df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


list_dfs, list_label_map, list_lr_order = [], [], []

SPEC_GENES_1 = ["Dlk1", "Notch"]

for gene in SPEC_GENES_1:
    df_liana_pop_gene = df_liana_pop[(df_liana_pop["ligand_complex"].str.contains(gene)) | (df_liana_pop["receptor_complex"].str.contains(gene))]
    df_plot, label_map, lr_order = get_df_pop(df_liana_pop_gene, POP_CHOICE, 
                                                specificity_rank_thresh=SPECIFICITY,
                                                magnitude_rank_thres=MAGNITUDE, 
                                                spec_weight_oe_thresh=SPEC_WEIGHT, 
                                                lr_logfc_thresh=LR_LOGFC, 
                                                circle_size_var=CIRCLE_SIZE,)

    list_dfs.append(df_plot)
    list_label_map.append(label_map)
    list_lr_order += [i for i in lr_order if i not in list_lr_order]    

SPEC_GENES_2 = ["Col"]


for gene in SPEC_GENES_2:
    df_liana_pop_gene = df_liana_pop[(df_liana_pop["ligand_complex"].str.contains(gene)) | (df_liana_pop["receptor_complex"].str.contains(gene))]
    df_plot, label_map, lr_order = get_df_pop(df_liana_pop_gene, POP_CHOICE, 
                                                specificity_rank_thresh=SPECIFICITY,
                                                magnitude_rank_thres=MAGNITUDE, 
                                                spec_weight_oe_thresh=SPEC_WEIGHT, 
                                                lr_logfc_thresh=LR_LOGFC, 
                                                circle_size_var=CIRCLE_SIZE,)

    list_dfs.append(df_plot)
    list_label_map.append(label_map)
    list_lr_order += [i for i in lr_order if i not in list_lr_order]    


SPEC_GENES_3 = ["Flt1"]


for gene in SPEC_GENES_3:
    df_liana_pop_gene = df_liana_pop[(df_liana_pop["ligand_complex"].str.contains(gene)) | (df_liana_pop["receptor_complex"].str.contains(gene))]
    df_plot, label_map, lr_order = get_df_pop(df_liana_pop_gene, POP_CHOICE, 
                                                specificity_rank_thresh=SPECIFICITY,
                                                magnitude_rank_thres=MAGNITUDE, 
                                                spec_weight_oe_thresh=SPEC_WEIGHT, 
                                                lr_logfc_thresh=LR_LOGFC, 
                                                circle_size_var=CIRCLE_SIZE,)

    list_dfs.append(df_plot)
    list_label_map.append(label_map)
    list_lr_order += [i for i in lr_order if i not in list_lr_order]    
    


plot_liana_pop(df_plot = pd.concat(list_dfs).drop_duplicates(),
                label_map = pd.concat(list_label_map).drop_duplicates(),
                lr_order = list_lr_order, 
                figsize=(6, 5)            )

plt.gca().axhline(y=1.5, color='#bcbcbc', linewidth=1)
plt.gca().axhline(y=10.5, color="#bcbcbc", linewidth=1)
plt.gca().xaxis.set_tick_params(rotation=90)
fig = plt.gcf()
savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}_SPECIFIC_GENES", fig_dir=FIG_DIR)



## Obtaining glial interactions

For practical purposes we are going to divide the glia into tSC and mSC first; then we are going to plot the interactions between tSCs and ENDO/PERI

In [ ]:
adata_joint_glia = adata_joint[adata_joint.obs["cell_type"].isin(["GLIA"])].copy()
sc.pp.filter_genes(adata_joint_glia, min_cells=3)
reprocess_in_place(adata_joint_glia , batch_key="gsm", n_comps=40, n_neighbors=25)

In [ ]:
dict_pops_glia = {"GLIA.tSC": ["Mcam", "Cspg4"], 
                  "GLIA.mSC": ["Mpz", "Prx"]}
assign_cats(adata_joint_glia, dict_pops_glia, key_added="cell_type")

In [ ]:
sc.pl.umap(adata_joint_glia, color=["cell_type", "S100b", "Mcam", "Cspg4", "Mpz", "Prx"], cmap=magma, legend_loc="on data", legend_fontsize=8, legend_fontoutline=2)

## GLIA.tSC ↔ ENDO y GLIA.tSC ↔ ENDO.PERI
Run after annotating adata_joint_glia. In the updated notebook the functions
are already defined. If you paste these cells into the original notebook,
import the functions from liana_population_plots_generalized.py first:

from liana_population_plots_generalized import (
    get_df_pop, get_df_pop_fixed_lr, plot_liana_pop,
    plot_general_case, plot_specific_case,
)

The liana_df_FAP_others tables already exclude these pairs. LIANA is recomputed
on the full object with the new glial labels. The original expression matrix is
kept and only the labels are transferred, matched by cell name.

In [ ]:
FOCAL_GLIA = ["GLIA.tSC"]
PARTNERS_GLIA = ["ENDO", "ENDO.PERI"]  # PERI is called ENDO.PERI in this notebook.

if not adata_joint.obs_names.is_unique or not adata_joint_glia.obs_names.is_unique:
    raise ValueError("Cell names must be unique in order to transfer the labels.")
if not adata_joint_glia.obs_names.isin(adata_joint.obs_names).all():
    raise ValueError("Some cells of adata_joint_glia are absent from adata_joint.")
glia_labels = adata_joint_glia.obs["cell_type"]
if glia_labels.isna().any() or not glia_labels.isin(["GLIA.tSC", "GLIA.mSC"]).all():
    raise ValueError("Check the glial labels: GLIA.tSC and GLIA.mSC are expected, with no missing values.")
if not adata_joint.obs.loc[glia_labels.index, "cell_type"].eq("GLIA").all():
    raise ValueError("The transfer expects those cells to be GLIA in the original object.")

adata_joint_lr_glia = adata_joint.copy()
adata_joint_lr_glia.obs["cell_type_lr"] = adata_joint_lr_glia.obs["cell_type"].astype(str)
adata_joint_lr_glia.obs.loc[glia_labels.index, "cell_type_lr"] = glia_labels.astype(str)
adata_joint_lr_glia.obs["cell_type_lr"] = (
    adata_joint_lr_glia.obs["cell_type_lr"].astype("category")
)
counts_glia_lr = adata_joint_lr_glia.obs["cell_type_lr"].value_counts()
print(counts_glia_lr.reindex(FOCAL_GLIA + PARTNERS_GLIA + ["GLIA.mSC", "GLIA"], fill_value=0))
missing_glia_lr = [name for name in FOCAL_GLIA + PARTNERS_GLIA
                   if counts_glia_lr.get(name, 0) == 0]
if missing_glia_lr:
    raise ValueError(f"Missing populations: {missing_glia_lr}. Check their exact names.")

In [ ]:
# Runs once. To change only filters or plots, repeat the next cell without
# recomputing LIANA. The permutations can take several minutes.
liana_df_glia = annotate_liana(
    run_liana(adata_joint_lr_glia, "cell_type_lr", n_perms=3000, n_jobs=8),
    adata_joint_lr_glia,
    "cell_type_lr",
)

In [ ]:
# Editable starting filters, close to those of the earlier figures.
GLIA_FILTERS = dict(
    specificity_rank_thresh=0.8,
    magnitude_rank_thres=0.8,
    spec_weight_oe_thresh=25,
    lr_logfc_thresh=1,
)

glia_plots = {}
for partner in PARTNERS_GLIA:
    df_plot_glia, labels_glia, order_glia = get_df_pop(
        liana_df_glia,
        population=partner,
        focal_populations=FOCAL_GLIA,
        circle_size_var="spec_weight_oe",
        **GLIA_FILTERS,
    )
    if df_plot_glia.empty:
        print(f"GLIA.tSC <-> {partner}: no LR passes the filters; check thresholds and coverage.")
    fig_glia = plot_liana_pop(
        df_plot_glia, labels_glia, order_glia,
        circle_size_var="spec_weight_oe",
        circle_hue_var="lr_logfc",
        size_lim=(5, 400), hue_lim=(-4, 4),
        figsize=(8, max(5, 0.3 * len(order_glia) + 2)),
    )
    fig_glia.suptitle(f"L–R interactions: GLIA.tSC ↔ {partner}", y=1.02)
    glia_plots[partner] = (df_plot_glia, labels_glia, order_glia, fig_glia)
    plt.show()
    # To save it with the usual helper:
    # savefig(fig=fig_glia, filename=f"4_LIANA_GLIA-tSC_{partner}", fig_dir=FIG_DIR)

Optional: a single panel with both partners and four directed columns.
Uncomment to produce that figure. LR pairs are selected if they pass the filter
in at least one column of the same direction. Bold marks the GLIA.tSC gene
(ligand when sending, receptor when receiving).

In [ ]:
# df_both, labels_both, order_both = get_df_pop(
#     liana_df_glia,
#     population=PARTNERS_GLIA,
#     focal_populations=FOCAL_GLIA,
#     **GLIA_FILTERS,
# )
# fig_both = plot_liana_pop(
#     df_both, labels_both, order_both,
#     figsize=(10, max(5, 0.3 * len(order_both) + 2)),
#     size_lim=(5, 400), hue_lim=(-4, 4),
# )
# fig_both.suptitle("L–R interactions: GLIA.tSC ↔ ENDO / ENDO.PERI", y=1.02)
# plt.show()